In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

tf.keras.utils.set_random_seed(42)

english = [
    "Norrköping is a city in Östergötland in eastern Sweden. It is built around the Motala ström river.",
    "Water power helped mills and factories grow along the river. The old industrial buildings are part of the city's history.",
    "The industrial area now includes cultural places. Arbetets museum is located in the well-known building called the Iron.",
    "Linköping University has a campus in Norrköping. Students and researchers contribute to the city.",
    "Trams are part of public transport. They connect several neighbourhoods in the city.",
    "Carl Johans park is a green space in the city centre. People can walk among its paths and trees.",
    "Kolmården Wildlife Park is a well-known attraction outside Norrköping.",
    "Norrköping combines industrial history, education, public transport, parks and culture."
]

swedish = [
    "Norrköping är en stad i Östergötland i östra Sverige. Staden ligger kring Motala ström.",
    "Vattenkraften bidrog till att kvarnar och fabriker växte fram längs ån. Industribyggnaderna är en del av stadens historia.",
    "Det gamla industriområdet har fått nya kulturella funktioner. Arbetets museum ligger i den välkända byggnaden Strykjärnet.",
    "Linköpings universitet har ett campus i Norrköping. Studenter och forskare bidrar till staden.",
    "Spårvagnar är en del av kollektivtrafiken. De förbinder flera stadsdelar.",
    "Carl Johans park är ett grönområde i centrum. Där kan människor promenera bland gångar och träd.",
    "Kolmårdens djurpark är ett välkänt besöksmål utanför Norrköping.",
    "Norrköping förenar industrihistoria, utbildning, kollektivtrafik, parker och kultur."
]

print("English:", len(english))
print("Swedish:", len(swedish))


English: 8
Swedish: 8


In [2]:
# Use the first 6 passages for training and the last 2 for testing.
english_train = english[:5]
english_val = english[5:6]
english_test = english[6:]

swedish_train = swedish[:5]
swedish_val = swedish[5:6]
swedish_test = swedish[6:]

vocabulary = sorted(set("".join(english + swedish).lower() + "\\n"))
char_to_id = {char: i for i, char in enumerate(vocabulary)}
vocab_size = len(vocabulary)

SEQUENCE_LENGTH = 40

def make_examples(passages):
    inputs, targets = [], []
    for passage in passages:
        text = passage.lower()
        ids = [char_to_id[c] for c in text]
        for i in range(len(ids) - SEQUENCE_LENGTH):
            inputs.append(ids[i:i + SEQUENCE_LENGTH])
            targets.append(ids[i + SEQUENCE_LENGTH])
    return np.array(inputs), np.array(targets)

X_en, y_en = make_examples(english_train)
X_en_val, y_en_val = make_examples(english_val)
X_en_test, y_en_test = make_examples(english_test)

X_sv, y_sv = make_examples(swedish_train)
X_sv_val, y_sv_val = make_examples(swedish_val)
X_sv_test, y_sv_test = make_examples(swedish_test)

print("Training examples — English:", len(X_en))
print("Training examples — Swedish:", len(X_sv))
print("Testing examples — English:", len(X_en_test))
print("Testing examples — Swedish:", len(X_sv_test))

Training examples — English: 320
Training examples — Swedish: 298
Testing examples — English: 77
Testing examples — Swedish: 68


In [3]:
#Build and train the two CNN models
def make_model():
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(SEQUENCE_LENGTH,)),
        tf.keras.layers.Embedding(vocab_size, 16),
        tf.keras.layers.Conv1D(128, 5, padding="causal", activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Conv1D(128, 5, padding="causal", activation="relu"),
        tf.keras.layers.Lambda(lambda x: x[:, -1, :]),
        tf.keras.layers.Dense(vocab_size)
    ])
    model.compile(
        optimizer="adam",
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )
    return model

# Give both models identical starting weights.
starter = make_model()
starting_weights = starter.get_weights()

english_model = make_model()
swedish_model = make_model()
english_model.set_weights(starting_weights)
swedish_model.set_weights(starting_weights)


english_history = english_model.fit(
    X_en, y_en, epochs=60, batch_size=32, verbose=1
)
swedish_history = swedish_model.fit(
    X_sv, y_sv, epochs=60, batch_size=32, verbose=1
)



Epoch 1/60


10/10 [==============================] - 3s 21ms/step - loss: 3.4801 - accuracy: 0.0750
Epoch 2/60
10/10 [==============================] - 0s 19ms/step - loss: 3.3923 - accuracy: 0.1562
Epoch 3/60
10/10 [==============================] - 0s 18ms/step - loss: 3.1479 - accuracy: 0.1500
Epoch 4/60
10/10 [==============================] - 0s 19ms/step - loss: 2.9355 - accuracy: 0.1500
Epoch 5/60
10/10 [==============================] - 0s 19ms/step - loss: 2.8716 - accuracy: 0.1781
Epoch 6/60
10/10 [==============================] - 0s 19ms/step - loss: 2.7928 - accuracy: 0.2094
Epoch 7/60
10/10 [==============================] - 0s 19ms/step - loss: 2.7193 - accuracy: 0.2156
Epoch 8/60
10/10 [==============================] - 0s 18ms/step - loss: 2.6299 - accuracy: 0.2281
Epoch 9/60
10/10 [==============================] - 0s 18ms/step - loss: 2.5427 - accuracy: 0.2500
Epoch 10/60
10/10 [==============================] - 0s 19ms/step - loss: 2.4483 - accuracy: 0.2750
Epoch

In [6]:
desktop = Path.home() / "Desktop"
desktop.mkdir(exist_ok=True)

english_model.save_weights(desktop / "norrkoping_english_cnn.weights.h5")
swedish_model.save_weights(desktop / "norrkoping_swedish_cnn.weights.h5")

print("Saved both model-weight files to:", desktop)
print("1. norrkoping_english_cnn.weights.h5")
print("2. norrkoping_swedish_cnn.weights.h5")

Saved both model-weight files to: C:\Users\wyn19\Desktop
1. norrkoping_english_cnn.weights.h5
2. norrkoping_swedish_cnn.weights.h5


In [8]:
import h5py

swedish_path = desktop / "norrkoping_swedish_cnn.weights.h5"

with h5py.File(swedish_path, "r") as f:

    def inspect(name, obj):
        if isinstance(obj, h5py.Dataset):
            print("\nLayer/weight:", name)
            print("Shape:", obj.shape)
            print("Values:")
            print(obj[()])
            print("-" * 50)

    f.visititems(inspect)


Layer/weight: layers\conv1d/vars/0
Shape: (5, 16, 128)
Values:
[[[ 0.26070052  0.02009695  0.2201482  ... -0.10788642  0.11519352
    0.13101214]
  [-0.00459023 -0.0939325   0.02268801 ...  0.13500421  0.18594816
    0.02564198]
  [ 0.07796603 -0.09635969 -0.06029063 ...  0.03573738  0.14549124
   -0.0653666 ]
  ...
  [-0.2528972  -0.01741859 -0.13620561 ...  0.02957053 -0.07862393
    0.06355587]
  [-0.156839   -0.15775488 -0.12342514 ... -0.01700324 -0.0594296
   -0.1541663 ]
  [ 0.15473449  0.16356371  0.05504086 ... -0.20726216  0.09543402
   -0.03803197]]

 [[ 0.20380491  0.04732084 -0.09792516 ...  0.0839267  -0.15331033
   -0.09597681]
  [ 0.11377062  0.24134512  0.10808115 ... -0.0694952  -0.0227635
    0.17478767]
  [ 0.08135829  0.15008399  0.10244148 ...  0.10678618  0.03808773
    0.03010657]
  ...
  [-0.12308599 -0.25144073  0.06530817 ...  0.03236625  0.16768895
   -0.11827961]
  [-0.03256925  0.0789168   0.0940023  ... -0.10829846 -0.20383291
   -0.01666986]
  [ 0.05367